# Objective

Profile response latency and token usage.

In [ ]:
#pip install -U langchain langchain-openai tiktoken pandas langchain-community

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

AZURE_OPENAI_KEY = os.getenv("AZURE_OPENAI_KEY")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_VERSION = os.getenv("AZURE_OPENAI_VERSION")


In [3]:
import time, pandas as pd
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.callbacks import get_openai_callback

model_name = "gpt-4o-mini"
TEMPERATURE =0.2

## Build a simple model, prompt and an output chain

In [31]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an assistant that answers in a concise manner. Keep your answers short and to the point."),
        ("human","{question}")
    ])
llm = AzureChatOpenAI(
    azure_endpoint= AZURE_OPENAI_ENDPOINT,
    azure_deployment=model_name,
    openai_api_key=AZURE_OPENAI_KEY,
    openai_api_version=AZURE_OPENAI_VERSION,
    temperature=TEMPERATURE)
chain = prompt | llm | StrOutputParser()

## Create a function to profile a single run

In [32]:
def profile_run(question:str):
    start = time.perf_counter()
    with get_openai_callback() as cb:
        answer = chain.invoke({"question": question})
        latency = time.perf_counter() - start
        stats = {
            "prompt_tokens": cb.prompt_tokens,
            "completion_tokens": cb.completion_tokens,
            "total_tokens": cb.total_tokens,
            "estimated_cost": cb.total_cost,
            "answer":answer
        }
    return answer,latency, stats

# Define benchmark test cases

In [33]:
test_cases = [
    {"label": "short factual", "question": "What is the capital of France?"},
    {"label": "medium reasoning", "question": "Explain the difference between concurrency and parallelism in 2-3 bullets."},
    {"label": "Structured output", "question": "Return a 3 item JSON array of best practices for python doctring."},
    {"label":"instruction rewrite","question":"Rewrite the following instruction to be more clear and concise: 'Can you please provide me with a list of the top 5 most popular programming languages in 2024, along with a brief description of each language and its primary use cases?'"},
]

In [36]:
rows = []
for test in test_cases:
    answer, latency, stats = profile_run(test["question"])
    print(f"Test: {test['label']}, Latency: {latency:.2f} seconds, Answer: {answer}")
    rows.append({
        "label": test["label"],
        "question": test["question"],
        "latency": latency,
        **stats
    })
df = pd.DataFrame(rows).sort_values("latency", ascending = True).reset_index(drop=True)

Test: short factual, Latency: 1.79 seconds, Answer: The capital of France is Paris.
Test: medium reasoning, Latency: 1.29 seconds, Answer: - **Concurrency** refers to the ability of a system to handle multiple tasks at the same time, allowing for overlapping execution but not necessarily simultaneous execution.
- **Parallelism** involves executing multiple tasks simultaneously, utilizing multiple processors or cores to perform computations at the same time.
- Concurrency is about structure and design, while parallelism is about execution and performance.
Test: Structured output, Latency: 2.08 seconds, Answer: ```json
[
    {
        "best_practice": "Use triple quotes for docstrings.",
        "description": "Always use triple double quotes (\"\"\") for multi-line docstrings."
    },
    {
        "best_practice": "Include a summary line.",
        "description": "Start with a brief summary of the function's purpose, followed by more detailed information if necessary."
    },
    {
   

In [37]:
df

,label,question,latency,prompt_tokens,completion_tokens,total_tokens,estimated_cost,answer
0,medium reasoning,Explain the difference between concurrency and...,1.291572,46,73,119,0.000051,- **Concurrency** refers to the ability of a s...
1,instruction rewrite,Rewrite the following instruction to be more c...,1.454329,79,28,107,0.000029,Please provide a list of the top 5 programming...
2,short factual,What is the capital of France?,1.794825,38,8,46,0.000010,The capital of France is Paris.
3,Structured output,Return a 3 item JSON array of best practices f...,2.079416,46,123,169,0.000081,"```json\n[\n {\n ""best_practice"": ""U..."


In [38]:
df.to_csv("outputs/response_latency.csv", index=False)